# Benchmark
Comparison of performance and accuracy between the methods in the `numerical_methods` library.

## Cell 1 – Imports

In [213]:
import numpy as np
import sympy as sp
from time import perf_counter
from scipy.integrate import quad, dblquad
from scipy.optimize._numdiff import approx_derivative
from scipy.optimize import brentq, approx_fprime

# Imported library already installed with pip.
import numerical_methods as nm

## Cell 2 – Helpers
Every section below times a method, compares it to a reference value, and prints one row.
These three functions do that job once so it isn't rewritten in each section.

In [214]:
def timed(func, *args, **kwargs):
    """Run func and return (result, elapsed_time_ms)."""
    start = perf_counter()
    result = func(*args, **kwargs)
    elapsed = (perf_counter() - start) * 1000
    return result, elapsed


def print_header(value_label="Result"):
    print(f"{'Method':30}{value_label:>15}{'Error':>15}{'Time (ms)':>15}")
    print("-" * 75)


def print_row(name, value, error, elapsed):
    """Print one formatted benchmark row. Pass value=None to show '—'."""
    value_str = f"{value:15.6f}" if value is not None else f"{'—':>15}"
    print(f"{name:30}{value_str}{error:15.2e}{elapsed:15.4f}")

## Cell 3 – Available methods
Only integration and differentiation methods are listed as dictionaries here: every function in each of these two families shares the same call signature (`method(f, a, b, n)` and `method(f, x, h)`), so a single loop can call any of them.

Root finding, series approximation, and linear algebra methods each take a different combination of arguments, so a shared dictionary would need one-off wrapping anyway — those calls are defined locally in their own cells, right next to the parameters they use.

In [215]:
integration_methods = {
    "Rectangle Rule": nm.rectangle_integrate,
    "Trapezoid Rule": nm.trapezoidal_integrate,
    "First Simpson Rule": nm.simpson1_integrate,
    "Second Simpson Rule": nm.simpson2_integrate,
    "Gauss-Legendre Quadrature": nm.gauss_legendre_integrate,
    "Monte Carlo": nm.monte_carlo_integrate,
}

differentiation_methods = {
    "Forward Difference": nm.fd.forward,
    "Backward Difference": nm.fd.backward,
    "Central Difference": nm.fd.central,
    "Central Difference nth": nm.fd.central_nth,
    "Richardson Method": nm.richardson_derivative,
}

## Cell 4 – Numerical Integration

In [216]:
a, b = -np.pi, np.pi
n = 120  # interval subdivisions (sample count for Monte Carlo)
f = lambda x: np.cos(x) ** 2 + np.sin(2 * x)

reference, elapsed_ref = timed(lambda: quad(f, a, b)[0])

print_header()
for name, method in integration_methods.items():
    result, elapsed = timed(method, f, a, b, n)
    print_row(name, result, abs(result - reference), elapsed)
print_row("SciPy (quad)", reference, 0.0, elapsed_ref)

F = lambda x, y: x**2 * y

reference, elapsed_ref = timed(lambda: dblquad(lambda y, x: F(x, y), 0, 1, lambda x: 0, lambda x: 2)[0])

result, elapsed = timed(nm.midpoint2_integrate, F, 0, 1, 0, 2)

print("")
print_header()
print_row("Midpoint Double", result, abs(result - reference), elapsed)
print_row("SciPy (dblquad)", reference, 0.0, elapsed_ref)

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Rectangle Rule                       3.141593       4.44e-16         0.1079
Trapezoid Rule                       3.141593       4.44e-16         0.2687
First Simpson Rule                   3.141593       4.44e-16         0.1379
Second Simpson Rule                  3.141593       4.44e-16         0.1227
Gauss-Legendre Quadrature            3.141593       2.09e-14         3.1906
Monte Carlo                          3.451981       3.10e-01         0.1569
SciPy (quad)                         3.141593       0.00e+00         0.1047

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Midpoint Double                      0.666650       1.67e-05         1.9486
SciPy (dblquad)                      0.666667       0.00e+00         0.2255


## Cell 5 – Numerical Differentiation

In [217]:
evaluation_point = 2
delta_x = 0.001

x_sym = sp.symbols('x')
f_sym = sp.cos(x_sym) ** 2 + sp.sin(2 * x_sym)
df_sym = sp.diff(f_sym, x_sym)
reference = float(df_sym.subs(x_sym, evaluation_point))

print_header()
for name, method in differentiation_methods.items():
    result, elapsed = timed(method, f, evaluation_point, delta_x)
    print_row(name, result, abs(result - reference), elapsed)

# SciPy: approx_fprime(x, f, h) -> gradient
result, elapsed = timed(approx_fprime, np.array([evaluation_point]), lambda v: f(v[0]), delta_x)
print_row("SciPy (approx_fprime)", result[0], abs(result[0] - reference), elapsed)

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Forward Difference                  -0.548317       2.17e-03         0.0728
Backward Difference                 -0.552652       2.17e-03         0.0066
Central Difference                  -0.550484       3.67e-07         0.0034
Central Difference nth              -0.550485       9.17e-08         0.0142
Richardson Method                   -0.550485       5.60e-14         0.0065
SciPy (approx_fprime)               -0.548317       2.17e-03         0.4247


## Cell 6 – Root Finding

In [218]:
# f has a sign change between -0.5 and -0.4 (bracket for Bisection / Ridders)
root_interval = (-0.5, -0.4)
x0_root = -0.45
max_iter = 100
tol = 1e-6

reference = float(sp.nsolve(f_sym, x_sym, x0_root))

root_methods = [
    ("Bisection Method", lambda: nm.bisection_calculate(f, *root_interval, tol)),
    ("Newton-Raphson Method", lambda: nm.newton_raphson_calculate(f, x0_root, max_iter, tol)[0]),
    ("Ridders Method", lambda: nm.ridders_calculate(f, *root_interval, max_iter, tol)[0]),
    ("Brent Method (SciPy)", lambda: brentq(f, *root_interval, xtol=tol)),
]

print_header()
for name, method in root_methods:
    root, elapsed = timed(method)
    print_row(name, root, abs(root - reference), elapsed)

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Bisection Method                    -0.463648       2.33e-07         0.0699
Newton-Raphson Method               -0.463648       3.92e-09         0.0254
Ridders Method                      -0.463648       1.81e-09         0.0200
Brent Method (SciPy)                -0.463648       1.59e-09         0.0571


## Cell 7 – Series Approximation

In [219]:
expansion_point = 0
order = 6
evaluation_x = 1.0
reference = f(evaluation_x)

series_methods = [
    ("Taylor Series", lambda: nm.taylor_approx(f, evaluation_x, expansion_point, order)),
    ("Fourier Series", lambda: nm.fourier_approx(f, evaluation_x, b, order)),
]

print_header()
for name, method in series_methods:
    result, elapsed = timed(method)
    print_row(name, result, abs(result - reference), elapsed)

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Taylor Series                        1.222223       2.10e-02         0.4598
Fourier Series                       1.201224       2.22e-16         1.8687


## Cell 8 – Linear Algebra

In [220]:
A = np.array([
    [10., 2., 1., 3., 0.],
    [2., 12., 2., 1., 4.],
    [1., 2., 15., 3., 2.],
    [3., 1., 3., 14., 5.],
    [0., 4., 2., 5., 13.]
])

b_vec = np.array([15., 20., 30., 25., 18.])

reference_solution = np.linalg.solve(A, b_vec)
reference_det = np.linalg.det(A)

# --- Linear system solve: Ax = b ---
solvers = [
    ("Linear System (Gauss)", lambda: nm.linearsystem_solve(A, b_vec, method="gauss")),
    ("Linear System (LU)", lambda: nm.linearsystem_solve(A, b_vec, method="lu")),
    ("Linear System (Cholesky)", lambda: nm.linearsystem_solve(A, b_vec, method="cholesky")),
    ("Linear System (QR)", lambda: nm.linearsystem_solve(A, b_vec, method="QR")),
    ("Linear System (NumPy)", lambda: np.linalg.solve(A, b_vec)),
]

print_header("Result (norm)")
for name, solver in solvers:
    x, elapsed = timed(solver)
    error = np.linalg.norm(x - reference_solution)
    print_row(name, np.linalg.norm(x), error, elapsed)

print("-" * 75)

# --- LU factorization: A = LU ---
(L, U), elapsed = timed(nm.lu_decomposition, A)
reconstruction_error = np.linalg.norm(L @ U - A)
print_row("LU Decomposition", None, reconstruction_error, elapsed)

print("-" * 75)

# --- Determinant ---
determinants = [
    ("Determinant (Gauss)", lambda: nm.determinant_calculate(A, method="gauss")),
    ("Determinant (LU)", lambda: nm.determinant_calculate(A, method="lu")),
    ("Determinant (NumPy)", lambda: np.linalg.det(A)),
]

for name, det_func in determinants:
    det, elapsed = timed(det_func)
    print_row(name, det, abs(det - reference_det), elapsed)

print("-" * 75)

# --- Jacobian ---
def F(v):
    x_, y_ = v
    return np.array([x_**2 + y_**2 - 4, x_ - y_])

point = np.array([1.0, 1.0])
h = 1e-6

# Separate symbols from the x_sym used in Cells 5-6 (scalar case) to avoid overwriting them.
x_j, y_j = sp.symbols('x y')
F_sym = sp.Matrix([x_j**2 + y_j**2 - 4, x_j - y_j])
reference_jacobian = np.array(
    F_sym.jacobian([x_j, y_j]).subs({x_j: point[0], y_j: point[1]})
).astype(float)

jacobians = [
    ("Jacobian", lambda: nm.jacobian_calculate(F, point, h)),
    ("Jacobian (SciPy)", lambda: approx_derivative(F, point)),
]

for name, jac_func in jacobians:
    J, elapsed = timed(jac_func)
    error = np.linalg.norm(J - reference_jacobian)
    print_row(name, None, error, elapsed)
    print(J)

Method                          Result (norm)          Error      Time (ms)
---------------------------------------------------------------------------
Linear System (Gauss)                2.329031       4.58e-16         0.1510
Linear System (LU)                   2.329031       3.85e-16         0.0669
Linear System (Cholesky)             2.329031       2.29e-16         0.1610
Linear System (QR)                   2.329031       1.12e-15         0.1159
Linear System (NumPy)                2.329031       0.00e+00         0.0465
---------------------------------------------------------------------------
LU Decomposition                            —       4.44e-16         0.0539
---------------------------------------------------------------------------
Determinant (Gauss)             209655.000000       4.07e-10         0.0764
Determinant (LU)                209655.000000       3.78e-10         0.0490
Determinant (NumPy)             209655.000000       0.00e+00         0.0301
------------